Seguindo o curso de Pandas do canal do YouTube Téo Me Why, o tópico principal agora é o tratamento de dados faltantes e duplicatas. Pensando nisso, para separar bem o aprendizado de uma área mais hands-on e também devido a importância do tratamento no dia a dia de um analista de dados, eu prefiri criar esse outro notebook focado apenas nesse processo.

Dataset utilizado (Kaggle): https://www.kaggle.com/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training

Esta é apenas uma primeira prática geral de limpeza e tratamento de dados, então focarei em buscar valores nulos e duplicados e decidir a melhor forma de lidar com eles mantendo o dataset estruturado. O meu objetivo é manter no mínimo 85% do dataset ao final do processo, permitindo que análises posteriores não sejam influenciadas por uma grande remoção de dados.

In [145]:
import pandas as pd
import numpy as np

O primeiro passo é importar os dados e analisar a quantidade inicial de linhas totais e faltantes e as colunas

In [146]:
df = pd.read_csv('../data/kaggle/dirty_cafe_sales.csv')
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [147]:
qtde_inicial = df.shape[0]
print(f'o dataset possui {df.shape[0]} linhas')
print('-'*40)
df.info()

o dataset possui 10000 linhas
----------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


A primeira tentativa de limpar os dados nulos é retirar as linhas que possuem todos os valores faltantes. Ao fazer o comando, foi constatado que nenhuma possuía esse padrão.

In [148]:
df = df.dropna(how='all')
print(f'Restaram {df.shape[0]} linhas')

Restaram 10000 linhas


Devido a falta de ações possíveis para identificar a data da transação e também o seu número de linhas nulas ser > 2.0% do dataset original, eu fiz a ação as transações que possuiam esse dado faltante.

In [149]:
df = df.dropna(how='any', subset=['Transaction Date'])
print(f'Restaram {df.shape[0]} linhas')

Restaram 9841 linhas


Após isso, decidi começar a analisar cada coluna do conjunto, nossas variáveis.

In [152]:
for variavel in df:
    print(variavel)
    print(f'a variável possui os {df[variavel].nunique()} valores únicos: ')
    print(df[variavel].unique())
    print()
    print('-'*40)

Transaction ID
a variável possui os 9841 valores únicos: 
<StringArray>
['TXN_1961373', 'TXN_4977031', 'TXN_4271903', 'TXN_7034554', 'TXN_3160411',
 'TXN_2602893', 'TXN_4433211', 'TXN_6699534', 'TXN_4717867', 'TXN_2064365',
 ...
 'TXN_1538510', 'TXN_3897619', 'TXN_2739140', 'TXN_4766549', 'TXN_7851634',
 'TXN_7672686', 'TXN_9659401', 'TXN_5255387', 'TXN_7695629', 'TXN_6170729']
Length: 9841, dtype: str

----------------------------------------
Item
a variável possui os 10 valores únicos: 
<StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',  'UNKNOWN',
 'Sandwich',        nan,    'ERROR',    'Juice',      'Tea']
Length: 11, dtype: str

----------------------------------------
Quantity
a variável possui os 7 valores únicos: 
<StringArray>
['2', '4', '5', '3', '1', 'ERROR', 'UNKNOWN', nan]
Length: 8, dtype: str

----------------------------------------
Price Per Unit
a variável possui os 8 valores únicos: 
<StringArray>
['2.0', '3.0', '1.0', '5.0', '4.0', '1.5', nan

Identificando que as seguintes variáveis (Item, Quantity, Price Per Unity, Total Spent, Payment Method, Location) possuem os valores "ERROR" e "UNKNOWN", eu transformo esses valores para todas as variáveis em NaN para que eu possa trabalhar de melhor maneira

In [153]:
replace = {
    'ERROR' : np.nan,
    'UNKNOWN' : np.nan
}

df= df.replace(replace)
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,NaN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


Antes de começar a pensar nas variáveis quantitativas em específico, eu decido remover as linhas do conjunto que possuem tanto o Item quanto o Preço por Unidade nulos, tendo em vista que o conjunto possui um cardápio pré-montado que não poderia ser utilizado caso ambos fossem nulos.

In [156]:
unidades = ['Item', 'Price Per Unit']
df = df.dropna(how='all', subset=unidades)
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9994,TXN_7851634,NaN,4,4.0,16.0,NaN,NaN,2023-01-08
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,NaN,2023-08-30
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


Agora que foram removidos os conflitos com o cardápio, eu peguei o dado desse cardápio disponível no kaggle e criei um dicionário para mapear e transformar os valores nulos de Item e Preço por Unidade em valores válidos.

Existe apenas um conflito, que é: Os preços de 'Sandwich' e 'Smoothie' são 4.0 e 'Cake' e 'Juice' são 3.0, logo, não como saber especificamente qual o item pedido, porém por convenção todos os item de 4.0 serão 'Sandwich' e todos os de 3.0 serão 'Cake'.

In [157]:
dict = {
    'Coffee' : '2.0',
    'Tea' : '1.5',
    'Sandwich' : '4.0',
    'Salad' : '5.0',
    'Cake' : '3.0',
    'Cookie' : '1.0',
    'Smoothie' : '4.0',
    'Juice' : '3.0',
    '2.0' : 'Coffee',
    '1.5' : 'Tea',
    '4.0' : 'Sandwich',
    '5.0' : 'Salad',
    '3.0' : 'Cake',
    '1.0' : 'Cookie'
}
df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Item'].map(dict))
df['Item'] = df['Item'].fillna(df['Price Per Unit'].map(dict))
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9994,TXN_7851634,Sandwich,4,4.0,16.0,NaN,NaN,2023-01-08
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,NaN,2023-08-30
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,1.0,3.0,Digital Wallet,NaN,2023-12-02


Agora, irei separar as variáveis qualitativas e quantitativas, desconsiderando as variáveis 'Transaction ID' e 'Transaction Date', devido a primeira não haver necessidade de tratamento e a segunda possuir uma tipagem especial que ocasionará em uma operação separada.

In [158]:
quantitativas = ['Quantity', 'Price Per Unit', 'Total Spent']
qualitativas = ['Item', 'Payment Method', 'Location']

Nesse primeiro momento, olharei para as variáveis quantitativas.

O primeiro passo é transformar elas para um tipo númerico, o float, para fazer operações;
O segundo passo é corrigir e aumentar o número de dados válidos da variável 'Total Spent', já que ela é o produto da Quantidade por Unidade.

In [161]:
df[quantitativas] = df[quantitativas].astype(float)
df['Total Spent'] = df['Quantity'] * df['Price Per Unit']
df['Quantity'] = df['Total Spent'] / df['Price Per Unit']
df.info()

<class 'pandas.DataFrame'>
Index: 9769 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    9769 non-null   str    
 1   Item              9769 non-null   str    
 2   Quantity          9322 non-null   float64
 3   Price Per Unit    9769 non-null   float64
 4   Total Spent       9322 non-null   float64
 5   Payment Method    6675 non-null   str    
 6   Location          5906 non-null   str    
 7   Transaction Date  9469 non-null   str    
dtypes: float64(3), str(5)
memory usage: 686.9 KB


Após corrigir o valor devido a disposição correta da variável Preço por Unidade advinda do cardápio, cabe retirar os valores nulos que ainda restam em Quantidade e Total Gasto, já que não possui mais ações possíveis para tornar válidos os dados nulos.

In [163]:
df = df.dropna(how='all', subset=['Quantity', 'Total Spent'])
print(f'Restaram {df.shape[0]} linhas')

Restaram 9322 linhas


Com isso, todas as variáveis quantitativas agora possuem o mesmo número de linhas, o que permite agora a análise das variáveis qualitativas

In [164]:
df[quantitativas].info()

<class 'pandas.DataFrame'>
Index: 9322 entries, 0 to 9999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Quantity        9322 non-null   float64
 1   Price Per Unit  9322 non-null   float64
 2   Total Spent     9322 non-null   float64
dtypes: float64(3)
memory usage: 291.3 KB


In [165]:
pct_final = (df.shape[0]/qtde_inicial) * 100
print(f'O data set possui {pct_final}% do original')

O data set possui 93.22% do original
